# Lab 1 — Neural Architecture Search su HW-NAS-Bench
## Spazio di ricerca, strategia di stima e algoritmi di ricerca semplici

In questo laboratorio lavoreremo con un **benchmark tabellare**: un file CSV estratto da **HW-NAS-Bench**, che contiene, per ogni architettura dello spazio di ricerca *NAS-Bench-201*, l'accuratezza ottenuta dopo l'addestramento e alcune metriche hardware (latenza, energia, ...) misurate su diversi dispositivi.

Avere un benchmark tabellare significa che **"addestrare e valutare" una rete costa una lettura di tabella**: possiamo quindi confrontare algoritmi di ricerca in pochi secondi, ripetendo gli esperimenti su molti seed, cosa impossibile se dovessimo addestrare davvero ogni rete.

**Obiettivi del lab**

1. Capire come è fatto uno **spazio di ricerca cell-based** e come si codifica un'architettura.
2. Separare le tre componenti di un sistema NAS: **search space**, **search strategy**, **performance estimation strategy**.
3. Implementare e confrontare **Grid Search**, **Random Search** e **Bayesian Optimization**.
4. Estendere la ricerca al caso **hardware-aware** (vincoli di latenza, fronte di Pareto).
5. *(Bonus)* Dare un primo sguardo a **algoritmi evolutivi**, **Reinforcement Learning** e **DARTS**.

Le celle marcate con ✏️ **ESERCIZIO** contengono un `# TODO` da completare. Le celle marcate con ❓ contengono domande a cui rispondere brevemente (anche a parole, in una cella di testo).

## 0. Setup

In [1]:
import itertools
import math
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("In Colab:", IN_COLAB)

ModuleNotFoundError: No module named 'numpy'

## 1. Caricamento dei dati

Il CSV viene **scaricato automaticamente** dall'indirizzo in `DATA_URL`: contiene una riga per ciascuna delle 15625 architetture, con la stringa dell'architettura, le accuratezze (da NAS-Bench-201, training di 200 epoche) e le metriche hardware di HW-NAS-Bench per CIFAR-10, CIFAR-100 e ImageNet16-120.

Le colonne usate nel lab sono indicate in `CONFIG`: potete cambiarle per lavorare su un altro dataset (es. `cifar100_test_acc`) o un altro dispositivo (es. `cifar10_raspi4_latency`, `cifar10_pixel3_latency`, `cifar10_fpga_latency`).

> 💡 Buona pratica: durante la ricerca si dovrebbe usare l'accuratezza di **validazione** (`cifar10_valid_acc`) e riportare alla fine quella di **test** della rete scelta. Per semplicità qui usiamo direttamente il test.

Se il file non viene trovato, viene generato un **dataset sintetico** con la stessa struttura, utile solo per provare il codice. ⚠️ I numeri sintetici non hanno alcun significato reale.

In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURAZIONE — modificare in base al proprio CSV
# ---------------------------------------------------------------------------
# URL del CSV fornito dal docente: non serve scaricare nulla
DATA_URL = "https://raw.githubusercontent.com/<UTENTE>/<REPO>/main/hw_nasbench201.csv.gz"

CONFIG = {
    "path": DATA_URL,
    "arch_col": "arch_str",                  # stringa NB201 dell'architettura
    "op_cols": None,
    "acc_col": "cifar10_test_acc",           # accuratezza da massimizzare
    "lat_col": "cifar10_edgegpu_latency",    # latenza (ms) su Jetson TX2
    "energy_col": "cifar10_edgegpu_energy",  # energia (mJ)
    "cost_col": "cifar10_train_time",        # tempo di training (s), per il costo simulato
    "use_synthetic_if_missing": True,
}

# Le 5 operazioni dello spazio NAS-Bench-201 (ordine fisso, usato per la codifica)
OPS = ["none", "skip_connect", "nor_conv_1x1", "nor_conv_3x3", "avg_pool_3x3"]
OP_SHORT = {"none": "zero", "skip_connect": "skip", "nor_conv_1x1": "c1x1",
            "nor_conv_3x3": "c3x3", "avg_pool_3x3": "pool"}
# I 6 archi della cella: (nodo_destinazione, nodo_sorgente)
EDGES = [(1, 0), (2, 0), (2, 1), (3, 0), (3, 1), (3, 2)]

In [ ]:
def parse_nb201(arch_str):
    # '|op~0|+|op~0|op~1|+|op~0|op~1|op~2|'  ->  tupla di 6 operazioni
    ops = []
    for node in arch_str.strip().split("+"):
        for tok in node.strip("|").split("|"):
            op, _src = tok.split("~")
            ops.append(op)
    assert len(ops) == 6, f"Stringa non valida: {arch_str}"
    return tuple(ops)


def to_nb201(ops):
    o = ops
    return (f"|{o[0]}~0|+|{o[1]}~0|{o[2]}~1|+"
            f"|{o[3]}~0|{o[4]}~1|{o[5]}~2|")


def make_synthetic(seed=0):
    # Dataset sintetico con la stessa struttura del benchmark (SOLO per test!)
    rng = np.random.default_rng(seed)
    gain = {"none": 0.0, "skip_connect": 1.0, "nor_conv_1x1": 2.5,
            "nor_conv_3x3": 4.0, "avg_pool_3x3": 0.8}
    lat = {"none": 0.0, "skip_connect": 0.05, "nor_conv_1x1": 0.6,
           "nor_conv_3x3": 1.6, "avg_pool_3x3": 0.3}
    edge_w = np.array([1.0, 0.6, 1.2, 0.5, 0.9, 1.3])
    rows = []
    for ops in itertools.product(OPS, repeat=6):
        g = np.array([gain[o] for o in ops]) * edge_w
        n_conv = sum(o.startswith("nor_conv") for o in ops)
        acc = 70 + 18 * (1 - np.exp(-g.sum() / 9)) - 0.6 * max(0, n_conv - 4) ** 2
        acc += rng.normal(0, 0.6)
        if not cell_is_connected(ops):
            acc = 10.0
        l = 2.0 + sum(lat[o] for o in ops) * (1 + rng.normal(0, 0.05))
        rows.append({"arch_str": to_nb201(ops),
                     "cifar10_test_acc": round(float(acc), 3),
                     "cifar10_edgegpu_latency": round(float(l), 4),
                     "cifar10_edgegpu_energy": round(float(l * 9 + rng.normal(0, 0.5)), 4),
                     "cifar10_train_time": round(float(800 + 400 * l + rng.normal(0, 30)), 1)})
    return pd.DataFrame(rows)


def cell_is_connected(ops):
    # Vero se esiste almeno un cammino input(0) -> output(3) senza archi 'none'
    e = [o != "none" for o in ops]
    n1 = e[0]
    n2 = e[1] or (e[2] and n1)
    n3 = e[3] or (e[4] and n1) or (e[5] and n2)
    return bool(n3)

In [ ]:
import os

def load_raw(cfg):
    path = cfg["path"]
    if path.startswith("http") or os.path.exists(path):
        try:
            print(f"Carico {path}")
            return pd.read_csv(path), False
        except Exception as ex:
            print(f"Impossibile leggere {path}: {ex}")
    if IN_COLAB and not path.startswith("http"):
        print(f"'{path}' non trovato: carica il CSV (oppure annulla per usare i dati sintetici).")
        try:
            from google.colab import files
            up = files.upload()
            if up:
                name = next(iter(up))
                return pd.read_csv(name), False
        except Exception as ex:  # upload annullato
            print("Upload non effettuato:", ex)
    if cfg["use_synthetic_if_missing"]:
        print("⚠️  Uso il DATASET SINTETICO (i numeri non sono reali).")
        return make_synthetic(), True
    raise FileNotFoundError(path)


def autodetect(df, cfg):
    cfg = dict(cfg)
    cols = list(df.columns)
    low = {c: c.lower() for c in cols}
    if cfg["arch_col"] is None and cfg["op_cols"] is None:
        for c in cols:
            v = df[c].dropna().astype(str)
            if len(v) and v.iloc[0].startswith("|") and "~" in v.iloc[0]:
                cfg["arch_col"] = c
                break
    if cfg["acc_col"] is None:
        cand = [c for c in cols if "acc" in low[c]]
        pref = [c for c in cand if "cifar10" in low[c] and "cifar100" not in low[c]]
        pref = [c for c in pref if "test" in low[c]] or pref or cand
        cfg["acc_col"] = pref[0] if pref else None
    if cfg["lat_col"] is None:
        cand = [c for c in cols if "lat" in low[c]]
        cfg["lat_col"] = cand[0] if cand else None
    if cfg["energy_col"] is None:
        cand = [c for c in cols if "energ" in low[c]]
        cfg["energy_col"] = cand[0] if cand else None
    if cfg["cost_col"] is None:
        cand = [c for c in cols if "time" in low[c] and "lat" not in low[c]]
        cfg["cost_col"] = cand[0] if cand else None
    return cfg


raw, IS_SYNTHETIC = load_raw(CONFIG)
CFG = autodetect(raw, CONFIG)
for k in ["arch_col", "op_cols", "acc_col", "lat_col", "energy_col", "cost_col"]:
    print(f"{k:>10}: {CFG[k]}")
assert CFG["acc_col"] is not None, "Specificare CONFIG['acc_col']"
assert CFG["arch_col"] or CFG["op_cols"], "Specificare arch_col oppure op_cols"

In [ ]:
# Costruzione della tabella "pulita" usata nel resto del notebook
if CFG["op_cols"]:
    ops_series = raw[CFG["op_cols"]].astype(str).apply(tuple, axis=1)
else:
    ops_series = raw[CFG["arch_col"]].astype(str).map(parse_nb201)

df = pd.DataFrame({"ops": ops_series})
for i in range(6):
    df[f"e{i}"] = df["ops"].map(lambda o, i=i: o[i])
df["acc"] = raw[CFG["acc_col"]].astype(float)
if df["acc"].max() <= 1.0:          # accuratezza in [0,1] -> percentuale
    df["acc"] *= 100
for name in ["lat", "energy", "cost"]:
    c = CFG[f"{name}_col"]
    df[name] = raw[c].astype(float) if c else np.nan

df = df.dropna(subset=["acc"]).drop_duplicates("ops").reset_index(drop=True)
N_ARCH = len(df)
ARCH2IDX = {o: i for i, o in enumerate(df["ops"])}
print(f"Architetture nel dataset: {N_ARCH}")
df.head()

## 2. Esplorazione del benchmark

Prima di cercare, guardiamo i dati: come sono distribuite le accuratezze? Quanto costa (in latenza) una rete accurata?

In [ ]:
print(df[["acc", "lat", "energy", "cost"]].describe().round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(df["acc"], bins=80, color="tab:blue")
ax[0].set(xlabel="Accuratezza (%)", ylabel="# architetture",
          title="Distribuzione dell'accuratezza")
if df["lat"].notna().any():
    ax[1].scatter(df["lat"], df["acc"], s=3, alpha=0.3)
    ax[1].set(xlabel=f"Latenza ({CFG['lat_col']})", ylabel="Accuratezza (%)",
              title="Accuratezza vs latenza")
plt.tight_layout(); plt.show()

best = df.loc[df["acc"].idxmax()]
print("Migliore architettura:", best["ops"], f"→ {best['acc']:.2f}%")
print("Percentile 99 dell'accuratezza:", np.percentile(df["acc"], 99).round(2))

### ❓ Domande — NAS vs Hyperparameter tuning

1. In che cosa differisce la ricerca di un'**architettura** dalla ricerca di **iperparametri** (learning rate, weight decay, batch size, ...)? Pensate al tipo di variabili (continue, discrete, categoriche, condizionali) e alla struttura dello spazio.
2. Nello spazio NAS-Bench-201 le variabili sono 6, ciascuna categorica con 5 valori. In che senso questo problema "assomiglia" a un problema di HP tuning? Cosa lo rende comunque un problema di NAS?
3. Guardando l'istogramma: che frazione di architetture è "buona"? Che implicazioni ha per una ricerca casuale?

## 3. Lo spazio di ricerca: *cell-based design*

In NAS-Bench-201 la rete è formata da uno **scheletro fisso** (stem, 3 stadi di celle impilate, blocchi di riduzione tra gli stadi, classificatore). La ricerca riguarda solo **la cella**, che viene poi ripetuta.

La cella è un **DAG con 4 nodi** (0 = input, 3 = output). Ogni coppia di nodi $i<j$ è collegata da un arco, per un totale di 6 archi, e su ogni arco si sceglie **una** operazione tra:

| operazione | significato |
|---|---|
| `none` | arco assente (zero) |
| `skip_connect` | identità |
| `nor_conv_1x1` | ReLU-Conv1×1-BN |
| `nor_conv_3x3` | ReLU-Conv3×3-BN |
| `avg_pool_3x3` | average pooling 3×3 |

Ogni nodo somma i contributi degli archi entranti.

In [ ]:
def draw_cell(ops, ax=None, title=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 3))
    xs = {0: 0, 1: 1, 2: 2, 3: 3}
    color = {"none": "lightgray", "skip_connect": "tab:green", "nor_conv_1x1": "tab:orange",
             "nor_conv_3x3": "tab:red", "avg_pool_3x3": "tab:blue"}
    for (dst, src), op in zip(EDGES, ops):
        span = dst - src
        rad = 0.0 if span == 1 else (-0.45 if span == 2 else 0.55)
        ls = ":" if op == "none" else "-"
        ax.annotate("", xy=(xs[dst], 0), xytext=(xs[src], 0),
                    arrowprops=dict(arrowstyle="-|>", color=color[op], lw=2, ls=ls,
                                    connectionstyle=f"arc3,rad={rad}",
                                    shrinkA=14, shrinkB=14))
        ylab = 0.08 if span == 1 else -rad * span / 2 + (0.07 if rad < 0 else -0.12)
        ax.text((xs[src] + xs[dst]) / 2, ylab, OP_SHORT[op], ha="center",
                fontsize=9, color=color[op])
    for n, x in xs.items():
        ax.add_patch(plt.Circle((x, 0), 0.13, color="white", ec="black", zorder=3))
        ax.text(x, 0, ["in", "1", "2", "out"][n], ha="center", va="center", zorder=4)
    ax.set_xlim(-0.4, 3.4); ax.set_ylim(-1.0, 0.7); ax.axis("off")
    ax.set_title(title or "")
    return ax

draw_cell(best["ops"], title=f"Migliore architettura ({best['acc']:.2f}%)")
plt.show()

### ✏️ ESERCIZIO 1 — Dimensione e struttura dello spazio

**1a.** Calcolate analiticamente la cardinalità dello spazio di ricerca e confrontatela con il numero di righe del CSV.

**1b.** Alcune celle **non collegano l'input all'output** (tutti i cammini contengono un arco `none`): qualunque sia il resto della rete, la cella produce zero. Completate la funzione `is_connected` e contate quante sono. Che accuratezza hanno nel benchmark?

In [ ]:
# 1a
# TODO: calcola la cardinalità teorica dello spazio (n. operazioni ^ n. archi)
raise NotImplementedError
print("Cardinalità teorica:", space_size, "| righe nel CSV:", N_ARCH)

In [ ]:
# 1b
def is_connected(ops):
    # ops: tupla di 6 operazioni nell'ordine di EDGES = [(1,0),(2,0),(2,1),(3,0),(3,1),(3,2)]
    # Restituisce True se esiste un cammino 0 -> 3 che non passa per archi 'none'.
    # TODO: propaga la "raggiungibilità" dal nodo 0 ai nodi 1, 2, 3
    raise NotImplementedError

df["connected"] = df["ops"].map(is_connected)
print("Celle scollegate:", (~df["connected"]).sum())
print(df.groupby("connected")["acc"].describe().round(2))

### ❓ Domande — Cell-based design
1. Quali sono i **vantaggi** di cercare una cella da ripetere invece di un'intera rete? E i **limiti**?
2. Lo spazio contiene architetture "duplicate" (diverse come stringa ma equivalenti come funzione calcolata)? Fate un esempio.

## 4. Le tre componenti di un sistema NAS

```
 ┌────────────────┐   architettura  ┌──────────────────────────────┐
 │ Search strategy│ ───────────────▶│ Performance estimation       │
 │  (GS, RS, BO,  │                 │ strategy (qui: lookup nel    │
 │   RL, EA, ...) │ ◀───────────────│ benchmark = training completo)│
 └────────────────┘    punteggio    └──────────────────────────────┘
          │ campiona da
          ▼
 ┌────────────────┐
 │  Search space  │  (cella NB201: 5^6 architetture)
 └────────────────┘
```

Nel nostro caso la **strategia di stima** è un *oracolo*: restituisce l'accuratezza finale come se avessimo addestrato la rete. Per rendere il confronto onesto, l'oracolo:

- conta le **query** e impone un **budget** (numero massimo di architetture valutabili);
- accumula il **costo simulato** (tempo di training, se presente nel CSV);
- può aggiungere **rumore** per simulare una stima a bassa fedeltà (es. poche epoche).

In [ ]:
class BudgetExhausted(Exception):
    pass


class Oracle:
    # Strategia di stima basata sul benchmark tabellare
    def __init__(self, df, budget, metric="acc", noise_std=0.0, seed=0):
        self.df, self.budget, self.metric = df, budget, metric
        self.noise_std = noise_std
        self.rng = np.random.default_rng(seed)
        self.true_vals = df[metric].to_numpy()
        self.history = []        # lista di (idx, valore_vero)
        self.cache = {}          # architetture già valutate (non si ripaga)
        self.sim_cost = 0.0

    @property
    def n_queries(self):
        return len(self.history)

    def remaining(self):
        return self.budget - self.n_queries

    def query(self, idx):
        idx = int(idx)
        if idx in self.cache:
            return self.cache[idx]
        if self.remaining() <= 0:
            raise BudgetExhausted()
        true = self.true_vals[idx]
        obs = true + (self.rng.normal(0, self.noise_std) if self.noise_std > 0 else 0.0)
        self.history.append((idx, true))
        self.cache[idx] = obs
        c = self.df["cost"].iat[idx]
        self.sim_cost += 0.0 if np.isnan(c) else c
        return obs

    def query_ops(self, ops):
        return self.query(ARCH2IDX[tuple(ops)])

    def incumbent_curve(self):
        # migliore accuratezza VERA trovata dopo ogni query
        vals = np.array([v for _, v in self.history])
        return np.maximum.accumulate(vals) if len(vals) else vals

    def best(self):
        i, v = max(self.history, key=lambda t: t[1])
        return self.df["ops"].iat[i], v


BEST_ACC = df["acc"].max()
o = Oracle(df, budget=3)
print(o.query(0), o.query(1), o.query(0), "| query usate:", o.n_queries)

## 5. Grid Search

La Grid Search enumera sistematicamente il prodotto cartesiano delle scelte. In NB201 la "griglia completa" coincide con l'intero spazio (15625 punti): con un budget di 100 valutazioni possiamo vederne meno dell'1%.

Due varianti tipiche quando il budget è limitato:
- **griglia completa troncata**: si enumera in ordine lessicografico e ci si ferma quando il budget finisce;
- **griglia ridotta**: si sceglie un sottoinsieme di operazioni per arco (es. 2 scelte → $2^6=64$ punti).

### ✏️ ESERCIZIO 2
Implementate `grid_search`. Usate `itertools.product`; saltate le combinazioni non presenti nel dataset; fermatevi quando il budget è esaurito.

In [ ]:
def grid_search(oracle, ops_per_edge=None):
    # ops_per_edge: lista di 6 liste di operazioni ammesse per ciascun arco
    #               (None = tutte le operazioni su tutti gli archi)
    if ops_per_edge is None:
        ops_per_edge = [OPS] * 6
    # TODO: enumera il prodotto cartesiano e interroga l'oracolo finché c'è budget
    raise NotImplementedError
    return oracle


BUDGET = 100
gs_full = grid_search(Oracle(df, BUDGET))
reduced = [["skip_connect", "nor_conv_3x3"]] * 6
gs_red = grid_search(Oracle(df, BUDGET), reduced)
gs_full_rev = grid_search(Oracle(df, BUDGET), [OPS[::-1]] * 6)
for name, o in [("GS completa", gs_full), ("GS completa (ordine inverso)", gs_full_rev),
                ("GS ridotta {skip,c3x3}", gs_red)]:
    ops, acc = o.best()
    print(f"{name:<30} query={o.n_queries:3d}  best={acc:.2f}%  regret={BEST_ACC-acc:.2f}")

### ❓ Domande — Grid Search
1. Perché la grid search "completa troncata" dipende così tanto dall'**ordine** delle operazioni? Quali archi vengono effettivamente esplorati con 100 query?
2. La griglia ridotta funziona bene o male? Da cosa dipende? Chi ha scelto le operazioni della griglia (e con quale conoscenza a priori)?

## 6. Random Search

La Random Search campiona architetture uniformemente (senza ripetizioni) fino a esaurire il budget. È sorprendentemente competitiva ed è il **baseline obbligatorio** di qualunque lavoro NAS.

### ✏️ ESERCIZIO 3
Implementate `random_search` campionando indici senza rimpiazzo.

In [ ]:
def random_search(oracle, seed=0):
    rng = np.random.default_rng(seed)
    # TODO: permuta gli indici e interroga l'oracolo finché c'è budget
    raise NotImplementedError
    return oracle


rs = random_search(Oracle(df, BUDGET), seed=0)
print("RS: best = %.2f%%, regret = %.2f" % (rs.best()[1], BEST_ACC - rs.best()[1]))

### ❓ Domanda
Sia $p$ la frazione di architetture nel top-1% ($p=0.01$). Qual è la probabilità che la Random Search con $n$ valutazioni ne trovi almeno una? Calcolatela per $n=100$ e verificate empiricamente con la cella seguente.

In [ ]:
thr = np.percentile(df["acc"], 99)
hits = [max(v for _, v in random_search(Oracle(df, BUDGET), seed=s).history) >= thr
        for s in range(200)]
print(f"Empirico: {np.mean(hits):.3f}   Teorico: {1 - 0.99**BUDGET:.3f}")

## 7. Bayesian Optimization

La BO costruisce un **modello surrogato** (qui un Gaussian Process) che predice l'accuratezza di architetture non ancora valutate, con una stima di **incertezza**. Una **funzione di acquisizione** decide quale architettura valutare dopo, bilanciando *exploitation* (media alta) ed *exploration* (incertezza alta).

**Codifica.** Il GP lavora su vettori: codifichiamo ogni architettura in **one-hot** (6 archi × 5 operazioni = 30 dimensioni).

**Expected Improvement.** Dato il miglior valore osservato $f^*$, media $\mu(x)$ e deviazione standard $\sigma(x)$:
$$
z = \frac{\mu(x) - f^* - \xi}{\sigma(x)}, \qquad
\mathrm{EI}(x) = (\mu(x) - f^* - \xi)\,\Phi(z) + \sigma(x)\,\varphi(z)
$$
con $\Phi$ e $\varphi$ CDF e PDF della normale standard ($\mathrm{EI}=0$ se $\sigma=0$).

Poiché lo spazio è piccolo, possiamo valutare l'acquisizione su **tutte** le architetture non ancora interrogate e prendere il massimo.

In [ ]:
def encode_onehot(ops):
    v = np.zeros(len(EDGES) * len(OPS))
    for e, op in enumerate(ops):
        v[e * len(OPS) + OPS.index(op)] = 1.0
    return v

X_ALL = np.stack(df["ops"].map(encode_onehot).to_numpy())
print("Matrice di codifica:", X_ALL.shape)


def make_gp(seed=0):
    kernel = (ConstantKernel(1.0, (1e-2, 1e3)) * Matern(length_scale=1.0, nu=2.5,
              length_scale_bounds=(1e-2, 1e2)) + WhiteKernel(1e-2, (1e-6, 1e1)))
    return GaussianProcessRegressor(kernel=kernel, normalize_y=True,
                                    n_restarts_optimizer=0, random_state=seed)

### ✏️ ESERCIZIO 4 — Expected Improvement

In [ ]:
def expected_improvement(mu, sigma, best_f, xi=0.01):
    # mu, sigma: array con media e deviazione standard predette dal GP
    # TODO: implementa la formula dell'EI (attenzione a sigma = 0)
    raise NotImplementedError

# Test veloce
print(expected_improvement(np.array([90., 90., 85.]), np.array([0.0, 1.0, 5.0]), best_f=90))

### ✏️ ESERCIZIO 5 — Il ciclo di BO

1. Valutate `n_init` architetture casuali.
2. Ripetete finché c'è budget: addestrate il GP sui punti osservati, predite media e deviazione standard su tutte le architetture non ancora valutate, scegliete quella con EI massimo e valutatela.

In [ ]:
def bayesian_optimization(oracle, n_init=10, xi=0.01, seed=0):
    rng = np.random.default_rng(seed)
    observed = list(rng.choice(N_ARCH, size=n_init, replace=False))
    y = [oracle.query(i) for i in observed]
    # TODO: ciclo fit GP -> predict -> EI -> argmax -> query
    raise NotImplementedError
    return oracle


t0 = time.time()
bo = bayesian_optimization(Oracle(df, BUDGET), seed=0)
print("BO: best = %.2f%%, regret = %.2f  (%.1fs)" % (bo.best()[1], BEST_ACC - bo.best()[1],
                                                      time.time() - t0))
draw_cell(bo.best()[0], title=f"Migliore trovata da BO ({bo.best()[1]:.2f}%)"); plt.show()

## 8. Confronto sperimentale

Gli algoritmi stocastici vanno confrontati su **più seed**. Riportiamo il **regret** (accuratezza migliore del benchmark − migliore trovata) in funzione del numero di valutazioni, in scala logaritmica (lineare vicino allo 0, così il regret nullo resta visibile).

⏱️ La BO è più lenta (un fit del GP per iterazione): con 10 seed e budget 100 servono circa 1–3 minuti.

In [ ]:
N_RUNS = 10

def run_many(algo, n_runs=N_RUNS, budget=BUDGET, **kw):
    curves = []
    for s in range(n_runs):
        o = algo(Oracle(df, budget, seed=s), seed=s, **kw)
        c = o.incumbent_curve()
        c = np.pad(c, (0, budget - len(c)), mode="edge")
        curves.append(BEST_ACC - c)
    return np.array(curves)

results = {}
results["Grid (completa)"] = np.array([BEST_ACC - grid_search(Oracle(df, BUDGET)).incumbent_curve()])
_c = grid_search(Oracle(df, BUDGET), reduced).incumbent_curve()
results["Grid (ridotta)"] = np.array([BEST_ACC - np.pad(_c, (0, BUDGET - len(_c)), mode="edge")])
results["Random Search"] = run_many(random_search)
t0 = time.time()
results["Bayesian Opt."] = run_many(bayesian_optimization)
print(f"BO: {time.time()-t0:.0f}s")

In [ ]:
def plot_regret(results, title="Regret vs numero di valutazioni"):
    plt.figure(figsize=(9, 5))
    x = np.arange(1, BUDGET + 1)
    for name, R in results.items():
        m = R.mean(0)
        plt.plot(x, m, label=name, lw=2)
        if len(R) > 1:
            se = R.std(0) / np.sqrt(len(R))
            plt.fill_between(x, m - se, m + se, alpha=0.2)
    plt.yscale("symlog", linthresh=0.1); plt.xlabel("# architetture valutate")
    plt.ylabel("Regret (punti %)"); plt.title(title); plt.legend(); plt.show()

plot_regret(results)
print(pd.DataFrame({k: {"regret finale (media)": v[:, -1].mean(),
                        "dev. std": v[:, -1].std()} for k, v in results.items()}).T.round(3))

### ❓ Domande — Confronto
1. Dopo quante valutazioni la BO supera stabilmente la Random Search? Perché all'inizio sono equivalenti?
2. Quanto *costerebbe* davvero (in GPU-ore) un esperimento con budget 100, se ogni rete richiedesse ~1–2 ore di training? Cosa ci permette il benchmark tabellare?
3. ✏️ Provate a cambiare `xi` nell'EI (es. 0, 0.1, 1.0) e `n_init` (es. 2, 30). Cosa succede?
4. ✏️ Ripetete la ricerca con un oracolo rumoroso (`Oracle(..., noise_std=2.0)`), che simula una stima a bassa fedeltà. Quale algoritmo soffre di più? Perché?

## 9. NAS hardware-aware

HW-NAS-Bench ci dà le metriche hardware: possiamo cercare la rete più accurata **che rispetti un vincolo di latenza**, oppure studiare il compromesso accuratezza/latenza tramite il **fronte di Pareto**.

Nota: misurare la latenza è (in questo setting) molto più economico che addestrare la rete, quindi possiamo assumere di conoscerla **gratis** per ogni architettura, prima di valutarla.

In [ ]:
def pareto_front(acc, lat):
    # Indici delle architetture non dominate (acc alta, latenza bassa)
    order = np.argsort(lat)
    front, best = [], -np.inf
    for i in order:
        if acc[i] > best:
            front.append(i); best = acc[i]
    return np.array(front)

if df["lat"].notna().any():
    pf = pareto_front(df["acc"].to_numpy(), df["lat"].to_numpy())
    plt.scatter(df["lat"], df["acc"], s=3, alpha=0.2, label="tutte")
    plt.plot(df["lat"].iloc[pf], df["acc"].iloc[pf], "r.-", label="fronte di Pareto")
    plt.xlabel("Latenza"); plt.ylabel("Accuratezza (%)"); plt.legend(); plt.show()
    print(f"Architetture sul fronte: {len(pf)}")
else:
    print("Nessuna colonna di latenza nel CSV: sezione non disponibile.")

### ✏️ ESERCIZIO 6 — Ricerca con vincolo di latenza

Scegliamo come vincolo la mediana della latenza. Modificate Random Search e BO in modo che **valutino solo architetture ammissibili** (latenza ≤ `LAT_MAX`). Confrontate il regret rispetto alla migliore architettura *ammissibile*.

Suggerimento: basta restringere l'insieme dei candidati.

In [ ]:
LAT_MAX = df["lat"].median()
FEASIBLE = np.flatnonzero(df["lat"].to_numpy() <= LAT_MAX)
BEST_FEAS = df["acc"].iloc[FEASIBLE].max()
print(f"Vincolo: latenza ≤ {LAT_MAX:.3f}  | ammissibili: {len(FEASIBLE)}  | best ammissibile: {BEST_FEAS:.2f}%")


def random_search_constrained(oracle, seed=0):
    rng = np.random.default_rng(seed)
    # TODO: campiona solo dagli indici in FEASIBLE
    raise NotImplementedError
    return oracle


def bo_constrained(oracle, n_init=10, xi=0.01, seed=0):
    rng = np.random.default_rng(seed)
    # TODO: come la BO, ma inizializzazione e candidati presi solo da FEASIBLE
    raise NotImplementedError
    return oracle

In [ ]:
def run_many_c(algo, n_runs=N_RUNS):
    out = []
    for s in range(n_runs):
        c = algo(Oracle(df, BUDGET, seed=s), seed=s).incumbent_curve()
        out.append(BEST_FEAS - np.pad(c, (0, BUDGET - len(c)), mode="edge"))
    return np.array(out)

res_c = {"RS vincolata": run_many_c(random_search_constrained),
         "BO vincolata": run_many_c(bo_constrained)}
plot_regret(res_c, f"Regret con vincolo latenza ≤ {LAT_MAX:.3f}")

### ❓ Domande — Hardware-aware NAS
1. La migliore architettura in assoluto è ammissibile? Quanta accuratezza "si perde" imponendo il vincolo?
2. Un'alternativa al vincolo rigido è una **funzione obiettivo scalarizzata**, ad esempio $\mathrm{acc}(x) - \lambda\,\mathrm{lat}(x)$ (approccio usato in MnasNet con una forma moltiplicativa). Quali pro e contro rispetto al vincolo?
3. *(Facoltativo)* Ripetete con la latenza di un altro dispositivo presente nel CSV. Le architetture ottime cambiano? Cosa ci dice questo sulla trasferibilità tra dispositivi?

## 10. Bonus — Uno sguardo ad altri algoritmi

### 10.1 Algoritmi evolutivi: *Regularized Evolution* (Real et al., 2019)

Si mantiene una **popolazione** di architetture. A ogni passo:
1. si estrae un **campione** casuale di `S` individui dalla popolazione (*tournament selection*);
2. il migliore del campione diventa **genitore** e viene **mutato** (si cambia l'operazione su un arco a caso);
3. il figlio viene valutato e inserito; l'individuo **più vecchio** viene rimosso (*aging*: questa è la "regolarizzazione").

In [ ]:
from collections import deque

def mutate(ops, rng):
    ops = list(ops)
    e = rng.integers(6)
    ops[e] = rng.choice([o for o in OPS if o != ops[e]])
    return tuple(ops)


def regularized_evolution(oracle, pop_size=20, sample_size=5, seed=0):
    rng = np.random.default_rng(seed)
    population = deque()
    # TODO: inizializza la popolazione a caso, poi ciclo tournament + mutazione + aging
    raise NotImplementedError
    return oracle


results["Reg. Evolution"] = run_many(regularized_evolution)
plot_regret(results)

### 10.2 Reinforcement Learning (Zoph & Le, 2017) — versione minimale

Un **controller** definisce una distribuzione sulle architetture; qui, per semplicità, è una distribuzione categorica indipendente per ciascun arco, con logit $\theta_{e,o}$ (nel lavoro originale è una RNN). Si campiona un'architettura, la si valuta, e si aggiornano i logit con **REINFORCE**:
$$
\theta \leftarrow \theta + \eta\,(R - b)\,\nabla_\theta \log p_\theta(\text{arch})
$$
dove $R$ è l'accuratezza e $b$ una baseline (media mobile) che riduce la varianza.

In [ ]:
def reinforce_search(oracle, lr=0.1, baseline_decay=0.9, seed=0):
    rng = np.random.default_rng(seed)
    theta = np.zeros((6, len(OPS)))
    baseline = None
    while oracle.remaining() > 0:
        p = np.exp(theta - theta.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
        choice = [rng.choice(len(OPS), p=p[e]) for e in range(6)]
        ops = tuple(OPS[c] for c in choice)
        R = oracle.query_ops(ops)
        baseline = R if baseline is None else baseline_decay * baseline + (1 - baseline_decay) * R
        adv = (R - baseline) / 10.0           # scala del reward (punti %)
        for e, c in enumerate(choice):        # grad log-softmax = onehot - p
            grad = -p[e]; grad[c] += 1
            theta[e] += lr * adv * grad
    return oracle

results["REINFORCE"] = run_many(reinforce_search)
plot_regret(results)

### 10.3 DARTS (Liu et al., 2019) — ricerca *differenziabile*

DARTS non campiona architetture discrete: su ogni arco mette **tutte** le operazioni in parallelo, pesate da un softmax su parametri continui $\alpha_{e,o}$ (*rilassamento continuo*):
$$
\bar o_e(x) = \sum_{o} \frac{\exp(\alpha_{e,o})}{\sum_{o'}\exp(\alpha_{e,o'})}\, o(x)
$$
I pesi della rete $w$ (sul training set) e gli $\alpha$ (sul validation set) si ottimizzano alternativamente con discesa del gradiente, addestrando **una sola supernet**. Alla fine si **discretizza** prendendo, per ogni arco, l'operazione con $\alpha$ massimo.

Non possiamo eseguire DARTS con un benchmark tabellare (servirebbe addestrare la supernet), ma possiamo simulare la fase di **discretizzazione**: dato un insieme di $\alpha$, quale architettura si ottiene e quanto vale?

In [ ]:
def discretize(alpha):
    # alpha: matrice 6 x 5 -> architettura (argmax per arco)
    return tuple(OPS[j] for j in alpha.argmax(1))

# Esempio: una supernet che (come spesso osservato in NB201) favorisce le skip connection
alpha = np.random.default_rng(0).normal(0, 0.1, (6, 5))
alpha[:, OPS.index("skip_connect")] += 0.5
arch = discretize(alpha)
print(arch, f"→ {df['acc'].iat[ARCH2IDX[arch]]:.2f}%")
draw_cell(arch, title="Cella ottenuta discretizzando alpha"); plt.show()

### ❓ Domande — Panoramica
1. Confrontate le curve di Regularized Evolution e REINFORCE con RS e BO. Con un budget di 100 valutazioni, quale approccio preferireste e perché?
2. Il costo di DARTS **non** si misura in "numero di architetture valutate". In che unità lo misurereste, e perché questo lo rende molto più economico dei metodi precedenti?
3. È noto che DARTS su NAS-Bench-201 tende a convergere verso celle piene di `skip_connect`. Guardate l'accuratezza di tali celle nel benchmark: perché è un problema? Che legame c'è tra la **strategia di stima** (pesi condivisi nella supernet) e questo fenomeno?

---
## Consegna
Consegnate il notebook con tutti gli esercizi completati e le risposte alle domande ❓ in celle di testo. Riportate nel testo il grafico finale di confronto e commentatelo in 5–10 righe.